### Custom modules

The mental model is very similar to creating a neural network in pytorch. Except that there is zeroing the gradients or backprop. The optimizer step is explored.

Only the LLM facing components will be optimized i.e. the __init__ where you would declare
all the layers of your NN. forward is just like the wiring(computation graph portion) 

In [1]:
import dspy
import os
from dotenv import load_dotenv

load_dotenv()

ANTHROPIC_KEY = os.getenv("ANTHROPIC_KEY")
OPENAI_KEY = os.getenv("OPENAI_KEY")

In [2]:
lm = dspy.LM("anthropic/claude-opus-4-6", api_key=ANTHROPIC_KEY)
dspy.configure(lm=lm, cache=False)

In [3]:
lm?

Signature:     
lm(
    prompt: str | None = None,
    messages: list[dict[str, typing.Any]] | None = None,
    **kwargs,
) -> list[dict[str, typing.Any] | str]
Type:           LM
String form:    <dspy.clients.lm.LM object at 0x7a3c09606840>
File:           ~/Desktop/DsPy_exploration/.venv/lib/python3.12/site-packages/dspy/clients/lm.py
Docstring:      A language model supporting chat or text completion requests for use with DSPy modules.
Init docstring:
Create a new language model instance for use with DSPy modules and programs.

Args:
    model: The model to use. This should be a string of the form ``"llm_provider/llm_name"``
           supported by LiteLLM. For example, ``"openai/gpt-4o"``.
    model_type: The type of the model, either ``"chat"`` or ``"text"``.
    temperature: The sampling temperature to use when generating responses.
    max_tokens: The maximum number of tokens to generate per response.
    cache: Whether to cache the model responses for reuse to improve performan

In [4]:
#Simple QA Module

class SimpleQAModule(dspy.Module):
    def __init__(self):
        self.answer = dspy.Predict("question -> answer")

    def forward(self, question):
        return self.answer(question=question)

qa_module = SimpleQAModule()
qa_module(question="Why is sky blue?")

Prediction(
    answer="The sky appears blue due to a phenomenon called **Rayleigh scattering**. Here's how it works:\n\n1. **Sunlight composition**: Sunlight (white light) is made up of all colors of the visible spectrum, each with different wavelengths. Blue and violet light have shorter wavelengths, while red and orange light have longer wavelengths.\n\n2. **Interaction with the atmosphere**: When sunlight enters Earth's atmosphere, it collides with gas molecules (mainly nitrogen and oxygen). These molecules are much smaller than the wavelengths of visible light.\n\n3. **Scattering**: Shorter wavelengths of light (blue and violet) are scattered much more efficiently by these tiny molecules than longer wavelengths (red, orange, yellow). Specifically, the amount of scattering is inversely proportional to the fourth power of the wavelength — meaning blue light (~470 nm) is scattered about 5.5 times more than red light (~650 nm).\n\n4. **Why blue and not violet?**: Even though violet li

In [5]:
## Chain 2 LLM calls

class JudgeQA(dspy.Module):
    def __init__(self):
        self.answer = dspy.Predict("question -> answer")
        self.judge = dspy.ChainOfThought("question, answer -> is_correct:bool, reasoning:str")

    def forward(self, question): #This is akin to foward(X)
        result = self.answer(question=question)
        judge_results = self.judge(question=question, answer=result)
        if judge_results.is_correct:
            return f"{result}\n\n{judge_results}"
        return f"Answer is wrong and reasoning is as follows {judge_results.reasoning}"

In [6]:
judge_qa = JudgeQA()
print(judge_qa(question="What is hard truth about life?")) #Subjective question. Dont worry. I am fine :)


Prediction(
    answer="There are many hard truths about life, but here are some of the most significant ones:\n\n1. **Life is not fair.** Bad things happen to good people, and good things happen to bad people. Merit alone doesn't guarantee success or happiness.\n\n2. **Everyone you love will either leave you or die.** Loss is an unavoidable part of the human experience. Every relationship ends — through distance, change, or death.\n\n3. **Time is limited and non-renewable.** You cannot get back a single moment. Every day that passes is gone forever, and none of us knows how many days we have left.\n\n4. **No one is coming to save you.** Ultimately, you are responsible for your own life. While others can help, the burden of building a meaningful life falls on your own shoulders.\n\n5. **Suffering is inevitable.** Pain — physical, emotional, existential — is woven into the fabric of being alive. You cannot avoid it; you can only learn how to navigate it.\n\n6. **Most people are too focu

#### Two-step email drafter
given a customer complaint, first analyze the complaint to identify the core issue and customer emotion, then draft a professional response.

Do both evaluation and optimization

In [7]:
complaints = [
    {"complaint": "I've been waiting 3 weeks for my refund and nobody responds to my emails. This is unacceptable.", "core_issue": "delayed_refund", "emotion": "frustrated"},
    {"complaint": "Your app deleted all my saved data after the last update. I lost 2 years of work.", "core_issue": "data_loss", "emotion": "angry"},
    {"complaint": "I was charged $200 for a feature I never signed up for. Please explain.", "core_issue": "unauthorized_charge", "emotion": "concerned"},
    {"complaint": "The delivery driver left my package in the rain and everything inside is ruined.", "core_issue": "damaged_delivery", "emotion": "upset"},
    {"complaint": "I cancelled my subscription last month but I'm still being billed. This is the third time I've reported this.", "core_issue": "billing_after_cancellation", "emotion": "frustrated"},
    {"complaint": "Your customer support chat told me my issue would be fixed in 24 hours. That was 10 days ago.", "core_issue": "broken_support_promise", "emotion": "disappointed"},
    {"complaint": "The product I received looks nothing like what was advertised on your website.", "core_issue": "misleading_product", "emotion": "disappointed"},
    {"complaint": "I've been a loyal customer for 8 years and you just removed the one feature I use daily without any notice.", "core_issue": "feature_removal", "emotion": "betrayed"},
]

In [8]:
from typing import Literal

full_dataset = [dspy.Example(complaint).with_inputs("complaint") for complaint in complaints]

class ComplaintAssesment(dspy.Signature):
    """ Assess a brief description of the complaint and categorize into 2 buckets
        1) core_issue - Helps understand the issue at hand
        2) Emotion - Helps understand the dissatisfaction customer had
    """
    complaint:str = dspy.InputField(desc="Summary of a complaint email")
    core_issue:Literal["delayed_refund", "data_loss", "unauthorized_charge", "damaged_delivery",\
        "billing_after_cancellation","broken_support_promise","misleading_product", "feature_removal"] = dspy.OutputField(desc="Issue described by customer. Choose the most appropriate categroy")
    emotion:Literal["frustrated", "angry", "concerned",\
        "upset", "disappointed", "betrayed"] = dspy.OutputField(desc="Emotion deduced based on complaint")

class EmailDraft(dspy.Signature):
    """Given a brief description of the customer complaint email, core issue and 
        their emotion, compose a draft reponse email that would help resolve the issue
    """

    complaint:str = dspy.InputField(desc="Summary of a complaint email")
    core_issue:str = dspy.InputField(desc="One word describing the core issue at hand")
    emotion:str = dspy.InputField(desc="Emotion potentially the customer had when writing this email")
    response_draft_email:str = dspy.OutputField(desc="A response email that would help resolve customer's complaint")


In [9]:
##The actual module
class EmailAnalyseAndDrafter(dspy.Module):

    def __init__(self):
        self.complaint_assesor = dspy.Predict(signature=ComplaintAssesment)
        self.email_draft = dspy.ChainOfThought(signature=EmailDraft)

    def forward(self, complaint):
        email_analysis_results = self.complaint_assesor(complaint=complaint)
        # return self.email_draft(complaint=complaint, core_issue=email_analysis_results.core_issue,
        #                     emotion=email_analysis_results.emotion
        #                        ), email_analysis_results # Apparently there is an "elegant way of renaming this instea dof sending tuples"
        draft = self.email_draft(complaint=complaint, core_issue=email_analysis_results.core_issue,
                                 emotion=email_analysis_results.emotion)
        return dspy.Prediction(
            core_issue=email_analysis_results.core_issue,
            emotion=email_analysis_results.emotion,
            draft_email=draft.response_draft_email
        ) #This also works as a dataclass object. No prediction is happening though
        

In [10]:
two_step_email = EmailAnalyseAndDrafter()
res = two_step_email(complaint=complaints[0]["complaint"])
res

Prediction(
    core_issue='delayed_refund',
    emotion='frustrated',
    draft_email='Subject: Re: Your Refund Request – Immediate Attention\n\nDear Valued Customer,\n\nThank you for reaching out, and I sincerely apologize for the unacceptable delay in both your refund and our response to your previous emails. I completely understand your frustration, and I want you to know that this is not the level of service we strive to provide.\n\nI have personally escalated your refund request to our finance team and can confirm that your refund is now being prioritized. You should see the funds returned to your original payment method within 3–5 business days from today. I will be monitoring this closely to ensure there are no further delays.\n\nTo make sure you are kept informed every step of the way, I will send you a confirmation email once the refund has been processed. If for any reason it has not appeared in your account within 5 business days, please reply directly to this email, and I 

In [11]:
## Evaluate the prediction

testset = full_dataset[0:7]

def evaluate_email_issue_emotion(ground_truth, prediction, trace=None ):
    return (prediction.core_issue.lower() == ground_truth.core_issue.lower()) and (
        prediction.emotion.lower() == ground_truth.emotion.lower())


email_evaluator = dspy.Evaluate(
    devset=testset,
    metric=evaluate_email_issue_emotion,
    display_progress=True,
    provide_traceback=True
)

email_evaluator(two_step_email)

Average Metric: 5.00 / 7 (71.4%): 100%|███████████████████████████████████████████████████████| 7/7 [00:00<00:00, 93.23it/s]

2026/04/10 15:00:29 INFO dspy.evaluate.evaluate: Average Metric: 5 / 7 (71.4%)


EvaluationResult(score=71.43, results=<list of 7 results>)

## Product review analyzer & response generator 
you receive a product review. Step 1: extract the product_aspect being discussed and the sentiment. Step 2: using that analysis, generate a public reply from the company.

In [12]:
reviews = [
    {"review": "The battery life is incredible, easily lasts 2 full days of heavy use", "product_aspect": "battery", "sentiment": "positive"},
    {"review": "Screen cracked after a minor drop, very disappointing build quality", "product_aspect": "durability", "sentiment": "negative"},
    {"review": "Camera quality is decent for the price but nothing special", "product_aspect": "camera", "sentiment": "neutral"},
    {"review": "Shipping took 3 weeks and the box arrived dented", "product_aspect": "shipping", "sentiment": "negative"},
    {"review": "The noise cancellation on these headphones is next level, worth every penny", "product_aspect": "audio", "sentiment": "positive"},
    {"review": "App keeps crashing when I try to sync my fitness data", "product_aspect": "software", "sentiment": "negative"},
    {"review": "Setup was plug and play, had it running in under 5 minutes", "product_aspect": "setup", "sentiment": "positive"},
    {"review": "Customer support took 4 days to respond and didn't even solve my issue", "product_aspect": "support", "sentiment": "negative"},
    {"review": "The design is sleek but it's a fingerprint magnet", "product_aspect": "design", "sentiment": "neutral"},
    {"review": "Overheats badly during gaming sessions, had to stop using it after 30 minutes", "product_aspect": "performance", "sentiment": "negative"},
]

In [13]:
full_dataset = [dspy.Example(review).with_inputs("review") for review in reviews]
train_set = full_dataset[0:4]
test_set = full_dataset[4:]

class AspectSentimentAnalyser(dspy.Signature):
    """ You are given a product review. Analyze the same and classify what aspect of
    the product is being commented on which is product_aspect and also deduce the sentiment
    associated with the review
    """
    review:str = dspy.InputField(desc="Product review")
    product_aspect:Literal["battery", "durability", "camera", "shipping", "audio", "software", "setup",\
        "support", "design", "performance"] = dspy.OutputField(desc="The aspect of the product the user is talking about. If there are multiple aspects, focus on the most criticized aspect")
    sentiment:Literal["positive", "negative", "neutral"] = dspy.OutputField(des="Sentiment associated with the review")

class PublicReplyGenerator(dspy.Signature):
    """You are given a review from a customer about our product and the product aspect 
    under scrutiny and the sentiment associated with the review. Your role is to generate
    a public reply to that review. Remember to be as professional and courteous as possible.
    Even if the review is extremely bad be nice. DO NOT offer any discounts"""

    review:str = dspy.InputField(desc="Product review")
    product_aspect = dspy.InputField(desc="Aspect of the product under critique")
    sentiment = dspy.InputField(desc="Sentiment associated with the review")

    reply = dspy.OutputField(desc="A public reply to the customer review.")


##Create the module

class ClassifierReviewWriter(dspy.Module):

    def __init__(self):
        self.classifier = dspy.Predict(signature=AspectSentimentAnalyser)
        self.reply_writer = dspy.Predict(signature=PublicReplyGenerator)

    def forward(self, review):
        classifier_results = self.classifier(review=review)
        reply = self.reply_writer(review=review, product_aspect=classifier_results.product_aspect,
                                 sentiment=classifier_results.sentiment)

        return dspy.Prediction(
            product_aspect = classifier_results.product_aspect,
            sentiment = classifier_results.sentiment,
            reply = reply.reply
        )


##Evaluation
def evaluate_aspect_sentiment(ground_truth, prediction, trace=None):
    return (ground_truth.product_aspect.lower() == prediction.product_aspect.lower()) and (
        ground_truth.sentiment.lower() == prediction.sentiment.lower())


review_evaluator = dspy.Evaluate(
    devset=test_set,
    metric=evaluate_aspect_sentiment,
    display_progress=True,
    provide_traceback=True
)

In [14]:
crw = ClassifierReviewWriter()
crw(review=full_dataset[0])

Prediction(
    product_aspect='battery',
    sentiment='positive',
    reply="Thank you so much for your wonderful review! We're thrilled to hear that the battery life has exceeded your expectations. Our engineering team worked hard to optimize battery performance, and it's fantastic to know that you're experiencing up to 2 full days of heavy use on a single charge. That's exactly the kind of reliability we strive to deliver. We truly appreciate you taking the time to share your experience, and we hope you continue to enjoy your product!"
)

In [15]:
review_evaluator(crw)

##Its already good with 100% accuracy - So no point of optimization

Average Metric: 6.00 / 6 (100.0%): 100%|█████████████████████████████████████████████████████| 6/6 [00:00<00:00, 112.15it/s]

2026/04/10 15:00:29 INFO dspy.evaluate.evaluate: Average Metric: 6 / 6 (100.0%)


EvaluationResult(score=100.0, results=<list of 6 results>)

### Constraining using Refine and BestofN

Note dspy.Assert has been deprecated. The intelligent version is "Refine".
Suggest has been deprecated and replaced by BestofN

Refine - Runs the module, checks a reward function and if the output doesnt pass, it feedsback detailed feedback and retries. Feedback generation is automatic



#### Answer the question in 30 words or fewer

Exceution will stop once the threshold has reached

In [16]:
class ConciseQA(dspy.Signature):
    """ Answer the question in 30 words or fewer. Only go above the bound if really required.
    Your buffer of safety is 40 words maximum. Be precise, accurate and complete"""

    question:str = dspy.InputField(desc="A general knowledge question")
    answer:str = dspy.OutputField(desc="A concise yet precise answer fewer than 30 words.")

base_qa = dspy.Predict(signature=ConciseQA)

##Refine 
### reward function
def reward_qa(args,pred):
    word_count = len(pred.answer.split())
    if word_count <= 30:
        return 1.0
    elif 30 < word_count <= 40:
        return 0.5
    return 0.0


refined_qa = dspy.Refine(
    module=base_qa,
    N=10, ##max iters
    reward_fn=reward_qa,
    threshold=1.0 
)
        

In [17]:
questions = [
    "What causes seasons on Earth?",
    "Why do planes fly?",
    "How does a vaccine work?",
    "What is blockchain?",
    "Why is the sky blue?",
]

for q in questions:
    result = refined_qa(question=q)
    print(f"Question - {q}")
    print(f"Answer - {result.answer} --- Word count {len(result.answer.split())}")

Question - What causes seasons on Earth?
Answer - Earth's ~23.5° axial tilt causes different hemispheres to receive varying angles and durations of sunlight as Earth orbits the Sun, producing seasons. --- Word count 22
Question - Why do planes fly?
Answer - Planes fly because their wings generate lift. Air moves faster over the curved top surface, creating lower pressure above the wing, producing an upward force exceeding gravity. --- Word count 27
Question - How does a vaccine work?
Answer - A vaccine introduces a weakened or inactive pathogen to stimulate the immune system to produce antibodies and memory cells, providing future protection against that disease. --- Word count 25
Question - What is blockchain?
Answer - Blockchain is a decentralized, distributed digital ledger that records transactions across multiple computers in a secure, transparent, and tamper-resistant way, ensuring data integrity without requiring a central authority. --- Word count 28
Question - Why is the sky b

In [18]:
dspy.inspect_history(n=1)





[2026-04-10T15:00:29.640793]

System message:

Your input fields are:
1. `question` (str): A general knowledge question
2. `hint_` (str): A hint to the module from an earlier run
Your output fields are:
1. `answer` (str): A concise yet precise answer fewer than 30 words.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## hint_ ## ]]
{hint_}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer the question in 30 words or fewer. Only go above the bound if really required.
        Your buffer of safety is 40 words maximum. Be precise, accurate and complete


User message:

[[ ## question ## ]]
Why is the sky blue?

[[ ## hint_ ## ]]
When answering questions, you MUST keep your answer to 30 words or fewer. Count your words carefully before responding. In this specific case, for 'Why is the sky blue?', a good answer would be: 'Rayleigh sc

In [19]:
new_lm = dspy.LM("anthropic/claude-opus-4-6", temperature=0.5, api_key=ANTHROPIC_KEY)
dspy.configure(lm=new_lm, cache=False)

### Format enforcement
Given a sentence answer in the format Subject | Predicate | Object
And
Each part in the format should contain under 5 words

In [20]:
##Create a signature

class FormatEnforcer(dspy.Signature):
    """ Given a factual statement you need to 
        1) Format into Subject | Predicate | Object pattern.
        2) Each component of the pattern i.e. Subject, Predicate and Object should be under 5 words
    """

    ip:str = dspy.InputField(desc="A factual question in English")
    op:str = dspy.OutputField(desc="A factual extracted pattern in specified format")

formatbase = dspy.Predict(signature=FormatEnforcer)

#Refine
def reward_triple(args, pred):
    parts = [p.strip() for p in pred.op.split("|")]
    if len(parts) != 3:
        return 0.0
    if any(len(p) == 0 for p in parts):
        return 0.0
    if all(len(p.split()) <= 5 for p in parts):
        return 1.0
    if all(len(p.split()) <= 8 for p in parts):
        return 0.5
    return 0.0

refiner = dspy.Refine(
    module = formatbase,
    N = 10,
    reward_fn = reward_triple,
    threshold=1.0
)


In [21]:
questions = [
    "What does the heart do?",
    "What is the role of mitochondria in a cell?",
    "What is the significance of the French Revolution?",
    "How does photosynthesis work?",
    "What happens during digestion?",
    "What is the relationship between supply and demand?",
    "What does the immune system do?",
    "Describe the water cycle.",
]

for q in questions:
    result = refiner(ip=q)
    print(f"Question - {q}")
    print(f"Formatted answer - {result.op}\n")
    

Question - What does the heart do?
Formatted answer - Heart | pumps | blood

Question - What is the role of mitochondria in a cell?
Formatted answer - Mitochondria | produce | energy for cells

Question - What is the significance of the French Revolution?
Formatted answer - French Revolution | signifies | political transformation

Question - How does photosynthesis work?
Formatted answer - Photosynthesis | converts sunlight into | chemical energy

Question - What happens during digestion?
Formatted answer - Food | is broken down into | nutrients

Question - What is the relationship between supply and demand?
Formatted answer - Supply and demand | determine | market prices

Question - What does the immune system do?
Formatted answer - Immune system | protects against | diseases and infections

Question - Describe the water cycle.
Formatted answer - Water | cycles through | evaporation and precipitation



In [22]:
dspy.inspect_history(n=1)





[2026-04-10T15:00:29.951628]

System message:

Your input fields are:
1. `ip` (str): A factual question in English
Your output fields are:
1. `op` (str): A factual extracted pattern in specified format
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## ip ## ]]
{ip}

[[ ## op ## ]]
{op}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given a factual statement you need to 
        1) Format into Subject | Predicate | Object pattern.
        2) Each component of the pattern i.e. Subject, Predicate and Object should be under 5 words


User message:

[[ ## ip ## ]]
Describe the water cycle.

Respond with the corresponding output fields, starting with the field `[[ ## op ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## op ## ]]
Water | cycles through | evaporation and precipitation

[[ ## completed ## ]]







In [23]:
##conclusion - dont see a point in Refine unless you are working on something where you the constraints cant be enforced with real world feedback.
# There is mostly no use for such constraints